# 03 — Test a Model's Intent Detection Against Your Labels

Once `data/my_labels.csv` has real labels, use this notebook to test how well a simple classifier (or an LLM prompt, if you have API access) predicts intent — then audit and categorize the errors, mirroring the JD's 'test customer service models... to confirm intent detection works as intended' responsibility.

In [ ]:
import json
import pandas as pd
from pathlib import Path

conversations = {c['conversation_id']: c for c in json.loads(Path('../data/conversations.json').read_text())}
labels = pd.read_csv('../data/my_labels.csv')
labels.head()

## Option A — Rule-based baseline (no API needed)

A simple keyword baseline to compare your labels against, useful as a sanity check even without model/API access.

In [ ]:
KEYWORD_RULES = {
    'billing_issue': ['charged', 'bill', 'billing'],
    'delivery_delay': ['where is my', 'delivery', 'shipped', 'shipment', 'delayed'],
    'refund_request': ['refund', 'money back'],
    'account_access': ['log in', 'login', 'password', 'locked out'],
    'product_complaint': ['broken', 'defective', 'damaged', 'cracked', 'wrong size', 'wrong item'],
}

def predict_intent(conv_id):
    text = ' '.join(t['text'] for t in conversations[conv_id]['turns'] if t['speaker'] == 'customer').lower()
    for intent, keywords in KEYWORD_RULES.items():
        if any(kw in text for kw in keywords):
            return intent
    return 'general_inquiry'

labels['predicted_intent'] = labels['conversation_id'].apply(predict_intent)
labels[['conversation_id', 'primary_intent', 'predicted_intent']]

In [ ]:
accuracy = (labels['primary_intent'] == labels['predicted_intent']).mean()
print(f'Baseline keyword-rule accuracy vs your labels: {accuracy:.2%}')

errors = labels[labels['primary_intent'] != labels['predicted_intent']]
errors[['conversation_id', 'primary_intent', 'predicted_intent']]

## Error analysis

For each mismatch above, note *why* the baseline got it wrong — e.g. ambiguous phrasing, missing keyword, intent stated indirectly. This write-up is the 'defect identification' deliverable — summarize the patterns you find.